# 🔠 Encoding ve Ölçeklendirme (Scaling)

Bu notebook, kategorik verileri makine öğrenmesi modellerinin anlayabileceği sayılara dönüştürme (Encoding) ve sayısal verileri aynı ölçeğe getirme (Scaling) şablonlarını içerir.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import category_encoders as ce

## 1. Klasik Encoding (OHE ve Label)

In [ ]:
def apply_label_encoding(df_train, df_test, cols):
    """
    Sadece Ağaç Tabanlı modellerde (XGBoost, Random Forest vb.) ve iki sınıflı/sıralı kategorilerde kullanılır.
    """
    df_train_encoded = df_train.copy()
    df_test_encoded = df_test.copy()
    
    for col in cols:
        le = LabelEncoder()
        df_train_encoded[col] = le.fit_transform(df_train_encoded[col])
        # Test setinde train'de olmayan bir kategori çıkarsa hata almamak için mapping yöntemi tercih edilebilir.
        # Şimdilik basit kullanım:
        df_test_encoded[col] = df_test_encoded[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
        le.classes_ = np.append(le.classes_, '<unknown>')
        df_test_encoded[col] = le.transform(df_test_encoded[col])
        
    return df_train_encoded, df_test_encoded

def apply_one_hot_encoding(df_train, df_test, cols):
    """
    Lineer modeller için en iyisidir. Her kategori için ayrı bir 0-1 sütunu oluşturur.
    """
    # get_dummies, hizalama(align) ile test setinde eksik çıkan kategorileri sıfırla doldurur.
    df_train_encoded = pd.get_dummies(df_train, columns=cols, drop_first=True)
    df_test_encoded = pd.get_dummies(df_test, columns=cols, drop_first=True)
    
    df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, join='left', axis=1, fill_value=0)
    return df_train_encoded, df_test_encoded

## 2. İleri Seviye Encoding (Kaggle Teknikleri)

Eğer bir sütunda çok fazla farklı kategori (High Cardinality) varsa OHE patlar. İşte o zaman bu yöntemler kullanılır.

In [ ]:
def apply_target_encoding(df_train, df_test, cols, target_col):
    """
    Target Encoding: Bir kategoriyi, o kategorideki hedef değişkenin ortalamasıyla değiştirir.
    Özellikle Ağaç modellerinde (LightGBM, XGBoost) çok etkilidir.
    Not: Aşırı öğrenmeyi (overfitting) önlemek için category_encoders kütüphanesi düzeltmeler (smoothing) yapar.
    """
    df_train_enc = df_train.copy()
    df_test_enc = df_test.copy()
    
    target_encoder = ce.TargetEncoder(cols=cols, smoothing=10)
    
    # Fit işlemi SADECE Train üzerinde yapılır (Data Leakage engellemek için)
    df_train_enc[cols] = target_encoder.fit_transform(df_train[cols], df_train[target_col])
    df_test_enc[cols] = target_encoder.transform(df_test[cols])
    
    return df_train_enc, df_test_enc

def apply_frequency_encoding(df_train, df_test, cols):
    """
    Frequency (Count) Encoding: Bir kategoriyi, veri setinde ne kadar sık geçtiği bilgisiyle (sayısıyla) değiştirir.
    Nadir görülen durumları ayırmak için iyidir.
    """
    df_train_enc = df_train.copy()
    df_test_enc = df_test.copy()
    
    for col in cols:
        freq = df_train[col].value_counts() / len(df_train)  # Yüzdesel sıklık (istersen sadece count da kullanabilirsin)
        df_train_enc[col] = df_train[col].map(freq)
        # Test setindekileri train'deki sıklıkla map'le. Eğer bilmediği bir kategori gelirse 0 yaz.
        df_test_enc[col] = df_test[col].map(freq).fillna(0)  
        
    return df_train_enc, df_test_enc

## 3. Ölçeklendirme (Scaling)

Lineer regresyon, SVM, KNN, PCA ve Sinir Ağları gibi modeller ölçeklendirme gerektirir. Ağaç tabanlı modeller (Random Forest, XGBoost) genellikle etkilenmez.

In [ ]:
def apply_scaling(df_train, df_test, cols, scaler_type='robust'):
    """
    scaler_type:
    - 'standard': Ortalama 0, Varyans 1 (Z-Score) -> Normal dağılımlar için iyidir.
    - 'robust'  : Medyan ve IQR kullanır -> Aykırı değerler (Outliers) varsa en iyisidir.
    - 'minmax'  : 0-1 arasına sıkıştırır -> Sinir ağları veya mesafe tabanlı modeller (KNN) için tercih edilebilir.
    """
    if scaler_type == 'robust':
        scaler = RobustScaler()
    elif scaler_type == 'standard':
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()
        
    df_train_scaled = df_train.copy()
    df_test_scaled = df_test.copy()
    
    # SADECE train'de fit ediyoruz.
    df_train_scaled[cols] = scaler.fit_transform(df_train[cols])
    df_test_scaled[cols] = scaler.transform(df_test[cols])
    
    return df_train_scaled, df_test_scaled